# Logistics Shipments Dataset 

## Data cleaning

### This notebook focuses on cleaning and preparing the logistics shipment dataset for analytical modeling.

### Data Quality Issues Identified

The exploratory analysis revealed the following issues:

- Date columns stored as text
- Numeric columns stored as strings
- Textual references in numeric fields (e.g. "See ASN-93 (ID#:1281)")
- Missing values in some columns

These issues will be addressed in the following steps.

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/shipments_raw.csv")

df.shape

#The dataset contains 10,324 shipment records.

(10324, 33)

### 2.1 Identifiying unique values in columns 'PQ First Sent to Client Date', 'PO Sent to Vendor Date'

In [2]:
df["PQ First Sent to Client Date"].unique()

<StringArray>
[   'Pre-PQ Process', 'Date Not Captured',          '11/18/09',
            '5/3/13',           '8/19/14',            '1/6/12',
           '2/22/13',          '10/28/14',           '2/20/13',
           '2/17/12',
 ...
           '11/1/11',           '1/16/12',           '3/16/13',
           '2/26/15',           '2/12/10',           '12/3/13',
          '12/19/14',            '8/2/11',            '2/2/12',
           '8/29/13']
Length: 765, dtype: str

In [3]:
df["PO Sent to Vendor Date"].unique()

<StringArray>
['Date Not Captured',          '11/13/06',           '12/1/06',
          '12/22/06',           '1/10/07',           '4/12/07',
           '5/13/07',           '5/17/07',           '7/13/07',
            '7/4/07',
 ...
           '1/31/12',            '6/3/14',           '8/11/10',
           '5/26/11',          '10/13/11',           '2/20/13',
           '9/26/12',           '12/3/13',            '3/9/10',
           '8/29/12']
Length: 897, dtype: str

### 2.2 Splitting Date, Process Status and Date Format

The columns "Scheduled Delivery Date", "Delivered to Client Date", "Delivery Recorded Date" stored as strings were converted to datetime format to enable temporal analysis.

The columns "PQ First Sent to Client Date" and "PO Sent to Vendor Date" contained both valid dates and textual process indicators such as "Pre-PQ Process", "Date Not Captured" and "N/A - From RDC".

To preserve this information:

- New datetime columns (pq_sent_to_client_date, po_sent_to_vendor_date) were created
- Separate categorical columns (pq_sent_to_client_status, po_sent_to_vendor_status) were created to retain non-date values

This allows distinguishing between actual timestamps and process states.

In [4]:
df["pq_sent_to_client_date"] = pd.to_datetime(
    df["PQ First Sent to Client Date"],
    errors="coerce"
)

df["po_sent_to_vendor_date"] = pd.to_datetime(
    df["PO Sent to Vendor Date"],
    errors="coerce"
)

C:\Users\danie\AppData\Local\Temp\ipykernel_14512\3375178345.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["pq_sent_to_client_date"] = pd.to_datetime(
C:\Users\danie\AppData\Local\Temp\ipykernel_14512\3375178345.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["po_sent_to_vendor_date"] = pd.to_datetime(


In [5]:
df["pq_sent_to_client_status"] = df["PQ First Sent to Client Date"].where(
    df["pq_sent_to_client_date"].isna(),
    None
)

df["po_sent_to_vendor_status"] = df["PO Sent to Vendor Date"].where(
    df["po_sent_to_vendor_date"].isna(),
    None
)

In [6]:
df["pq_sent_to_client_status"] = df["pq_sent_to_client_status"].replace({
    "Date Not Captured": "Missing",
})

df["po_sent_to_vendor_status"] = df["po_sent_to_vendor_status"].replace({
    "Date Not Captured": "Missing",
    "N/A - From RDC": "Not Applicable - From RDC"
})

In [7]:
df["Scheduled Delivery Date"] = pd.to_datetime(
    df["Scheduled Delivery Date"],
    errors="coerce"
)

df["Delivered to Client Date"] = pd.to_datetime(
    df["Delivered to Client Date"],
    errors="coerce"
)

df["Delivery Recorded Date"] = pd.to_datetime(
    df["Delivery Recorded Date"],
    errors="coerce"
)

C:\Users\danie\AppData\Local\Temp\ipykernel_14512\3769262772.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Scheduled Delivery Date"] = pd.to_datetime(
C:\Users\danie\AppData\Local\Temp\ipykernel_14512\3769262772.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Delivered to Client Date"] = pd.to_datetime(
C:\Users\danie\AppData\Local\Temp\ipykernel_14512\3769262772.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Delivery Recorded Date"] = pd.to_datetime(


In [8]:
# Validation of date columns after conversion
df[
    ["pq_sent_to_client_date", "pq_sent_to_client_status", "po_sent_to_vendor_date", 
     "po_sent_to_vendor_status", "Scheduled Delivery Date", "Delivered to Client Date", 
     "Delivery Recorded Date"]
].info()

<class 'pandas.DataFrame'>
RangeIndex: 10324 entries, 0 to 10323
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   pq_sent_to_client_date    7643 non-null   datetime64[us]
 1   pq_sent_to_client_status  2681 non-null   str           
 2   po_sent_to_vendor_date    4592 non-null   datetime64[us]
 3   po_sent_to_vendor_status  5732 non-null   str           
 4   Scheduled Delivery Date   10324 non-null  datetime64[us]
 5   Delivered to Client Date  10324 non-null  datetime64[us]
 6   Delivery Recorded Date    10324 non-null  datetime64[us]
dtypes: datetime64[us](5), str(2)
memory usage: 564.7 KB


### 2.3 Cleaning Numeric Columns

Some numeric columns contained non-numeric text values such as:

- "See ASN-93 (ID#:1281)"
- "Weight Captured Separately"
- "Freight Included in Commodity Cost"
- "Invoiced Separately"

These entries indicate that the corresponding values were either recorded
in a different record or handled outside the dataset.

In [9]:
# Convert weight and freight cost to numeric, coercing errors to NaN
df["weight_kg"] = pd.to_numeric(
    df["Weight (Kilograms)"],
    errors="coerce"
)
df["freight_cost_usd"] = pd.to_numeric(
    df["Freight Cost (USD)"],
    errors="coerce"
)

# Create details columns for weight and freight cost where original values are preserved if conversion failed
df["weight_kg_details"] = df["Weight (Kilograms)"].where(
    df["weight_kg"].isna(),
    None
)
df["freight_cost_usd_details"] = df["Freight Cost (USD)"].where(
    df["freight_cost_usd"].isna(),
    None
)

df[
    ["weight_kg", "freight_cost_usd", "weight_kg_details", "freight_cost_usd_details"]
].info()

<class 'pandas.DataFrame'>
RangeIndex: 10324 entries, 0 to 10323
Data columns (total 4 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   weight_kg                 6372 non-null   float64
 1   freight_cost_usd          6198 non-null   float64
 2   weight_kg_details         3952 non-null   str    
 3   freight_cost_usd_details  4126 non-null   str    
dtypes: float64(2), str(2)
memory usage: 322.8 KB


In [10]:
df[["ID",
     "Weight (Kilograms)", "weight_kg", "weight_kg_details",
     "Freight Cost (USD)", "freight_cost_usd", "freight_cost_usd_details"]].tail(10)

,ID,Weight (Kilograms),weight_kg,weight_kg_details,Freight Cost (USD),freight_cost_usd,freight_cost_usd_details
10314,86813,See DN-4274 (ID#:84472),NaN,See DN-4274 (ID#:84472),See DN-4274 (ID#:84472),NaN,See DN-4274 (ID#:84472)
10315,86814,15198,15198.0,NaN,26180,26180.0,NaN
10316,86815,1547,1547.0,NaN,3410,3410.0,NaN
10317,86816,See DN-4282 (ID#:83919),NaN,See DN-4282 (ID#:83919),See DN-4282 (ID#:83919),NaN,See DN-4282 (ID#:83919)
10318,86817,See DN-4307 (ID#:83920),NaN,See DN-4307 (ID#:83920),See DN-4307 (ID#:83920),NaN,See DN-4307 (ID#:83920)
10319,86818,See DN-4307 (ID#:83920),NaN,See DN-4307 (ID#:83920),See DN-4307 (ID#:83920),NaN,See DN-4307 (ID#:83920)
10320,86819,See DN-4313 (ID#:83921),NaN,See DN-4313 (ID#:83921),See DN-4313 (ID#:83921),NaN,See DN-4313 (ID#:83921)
10321,86821,Weight Captured Separately,NaN,Weight Captured Separately,Freight Included in Commodity Cost,NaN,Freight Included in Commodity Cost
10322,86822,1392,1392.0,NaN,Freight Included in Commodity Cost,NaN,Freight Included in Commodity Cost
10323,86823,Weight Captured Separately,NaN,Weight Captured Separately,Freight Included in Commodity Cost,NaN,Freight Included in Commodity Cost


For analytical purposes

- The columns `weight_kg` and `freight_cost_usd` were converted
  to numeric format, coercing invalid entries to NaN.

- Original non-numeric values were preserved in separate columns:
  `weight_kg_details` and `freight_cost_usd_details`.

This allows identifying relationships between records without introducing
assumptions by automatically imputing values.

### 2.4 Resolving Cross-Referenced Weight and Freight Cost Values

Several rows in `Weight (Kilograms)` and `Freight Cost (USD)` contained text 
references instead of numeric values, such as `"See ASN-4332 (ID#:1115)"`.

These references point to another shipment record (by ID) where the actual 
value was recorded. A regex lookup-join strategy was applied to recover 
these values before dropping any rows.

In [11]:
import re

def extract_reference_id(value):
    match = re.search(r"ID#:(\d+)", str(value))
    return int(match.group(1)) if match else None

df["reference_id_weight"]       = df["weight_kg_details"].apply(extract_reference_id)
df["reference_id_freight_cost"] = df["freight_cost_usd_details"].apply(extract_reference_id)

df[
    df["reference_id_weight"].notna()
]

,ID,Project Code,PQ #,PO / SO #,ASN/DN #,Country,Managed By,Fulfill Via,Vendor INCO Term,Shipment Mode,...,pq_sent_to_client_date,po_sent_to_vendor_date,pq_sent_to_client_status,po_sent_to_vendor_status,weight_kg,freight_cost_usd,weight_kg_details,freight_cost_usd_details,reference_id_weight,reference_id_freight_cost
8,46,112-NG-T01,Pre-PQ Process,SCMS-156,ASN-93,Nigeria,PMO - US,Direct Drop,EXW,Air,...,NaT,NaT,Pre-PQ Process,Missing,NaN,NaN,See ASN-93 (ID#:1281),See ASN-93 (ID#:1281),1281.0,1281.0
86,961,108-VN-T01,Pre-PQ Process,SCMS-37170,ASN-3562,Vietnam,PMO - US,Direct Drop,EXW,Air,...,NaT,2009-01-16,Pre-PQ Process,NaN,NaN,NaN,See ASN-3562 (ID#:960),See ASN-3562 (ID#:960),960.0,960.0
94,1047,106-HT-T01,Pre-PQ Process,SCMS-40000,ASN-3675,Haiti,PMO - US,Direct Drop,CIP,Air,...,NaT,2009-03-06,Pre-PQ Process,NaN,NaN,NaN,See ASN-3675 (ID#:1046),See ASN-3675 (ID#:1046),1046.0,1046.0
140,1299,107-RW-T01,Pre-PQ Process,SCMS-268,ASN-242,Rwanda,PMO - US,Direct Drop,EXW,Air,...,NaT,2006-12-22,Pre-PQ Process,NaN,NaN,NaN,See ASN-242 (ID#:64),See ASN-242 (ID#:64),64.0,64.0
141,1300,107-RW-T01,Pre-PQ Process,SCMS-268,ASN-242,Rwanda,PMO - US,Direct Drop,EXW,Air,...,NaT,2006-12-22,Pre-PQ Process,NaN,NaN,NaN,See ASN-242 (ID#:64),See ASN-242 (ID#:64),64.0,64.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10314,86813,151-NG-T30,FPQ-14989,SO-51422,DN-4274,Nigeria,PMO - US,From RDC,N/A - From RDC,Air Charter,...,2014-09-19,NaT,NaN,Not Applicable - From RDC,NaN,NaN,See DN-4274 (ID#:84472),See DN-4274 (ID#:84472),84472.0,84472.0
10317,86816,151-NG-T30,FPQ-16313,SO-51440,DN-4282,Nigeria,PMO - US,From RDC,N/A - From RDC,Air,...,2015-05-04,NaT,NaN,Not Applicable - From RDC,NaN,NaN,See DN-4282 (ID#:83919),See DN-4282 (ID#:83919),83919.0,83919.0
10318,86817,103-ZW-T30,FPQ-15197,SO-50020,DN-4307,Zimbabwe,PMO - US,From RDC,N/A - From RDC,Truck,...,2014-10-16,NaT,NaN,Not Applicable - From RDC,NaN,NaN,See DN-4307 (ID#:83920),See DN-4307 (ID#:83920),83920.0,83920.0
10319,86818,103-ZW-T30,FPQ-15197,SO-50020,DN-4307,Zimbabwe,PMO - US,From RDC,N/A - From RDC,Truck,...,2014-10-16,NaT,NaN,Not Applicable - From RDC,NaN,NaN,See DN-4307 (ID#:83920),See DN-4307 (ID#:83920),83920.0,83920.0


In [12]:
lookup = df[["ID", "weight_kg", "freight_cost_usd"]].copy()

In [13]:
weight_ref = (
    lookup[["ID", "weight_kg"]]
    .rename(columns={"ID": "ref_id", "weight_kg": "weight_kg_ref"})
)

df = df.merge(
    weight_ref,
    left_on="reference_id_weight",
    right_on="ref_id",
    how="left"
).drop(columns=["ref_id"])

df["weight_kg_final"] = df["weight_kg"].combine_first(df["weight_kg_ref"])

In [14]:
freight_ref = (
    lookup[["ID", "freight_cost_usd"]]
    .rename(columns={"ID": "ref_id", "freight_cost_usd": "freight_cost_ref"})
)

df = df.merge(
    freight_ref,
    left_on="reference_id_freight_cost",
    right_on="ref_id",
    how="left"
).drop(columns=["ref_id"])

df["freight_cost_usd_final"] = df["freight_cost_usd"].combine_first(df["freight_cost_ref"])


In [23]:
print("=== weight_kg ===")
print(f"  Original values  : {df['weight_kg'].notna().sum()}")
print(f"  Final values     : {df['weight_kg_final'].notna().sum()}")
print(f"  Recovered values : {df['weight_kg_final'].notna().sum() - df['weight_kg'].notna().sum()}")

print("\n=== freight_cost_usd ===")
print(f"  Original values  : {df['freight_cost_usd'].notna().sum()}")
print(f"  Final values     : {df['freight_cost_usd_final'].notna().sum()}")
print(f"  Recovered values : {df['freight_cost_usd_final'].notna().sum() - df['freight_cost_usd'].notna().sum()}")

unresolved_weight = df[
    df["reference_id_weight"].notna() & df["weight_kg_final"].isna()
][["ID", "weight_kg_details", "reference_id_weight", "weight_kg_final"]]

print(f"\n⚠ Referencces not solved: {len(unresolved_weight)}")
print(unresolved_weight)

=== weight_kg ===
  Original values  : 6372
  Final values     : 8691
  Recovered values : 2319

=== freight_cost_usd ===
  Original values  : 6198
  Final values     : 8538
  Recovered values : 2340

⚠ Referencces not solved: 126
          ID        weight_kg_details  reference_id_weight  weight_kg_final
257     2481  See ASN-4332 (ID#:1115)               1115.0              NaN
629     6400  See ASN-3635 (ID#:3745)               3745.0              NaN
631     6402  See ASN-3760 (ID#:5049)               5049.0              NaN
643     6437  See ASN-4332 (ID#:1115)               1115.0              NaN
644     6438  See ASN-4332 (ID#:1115)               1115.0              NaN
...      ...                      ...                  ...              ...
10040  86442  See DN-2499 (ID#:83004)              83004.0              NaN
10172  86585  See DN-3230 (ID#:86584)              86584.0              NaN
10176  86589  See DN-3260 (ID#:82593)              82593.0              NaN
10188  86

### Recovery Results

| Column            | Original values | Final values | Recovered |
|-------------------|-----------------|--------------|-----------|
| `weight_kg`       | 6,372           | 8,691        | +2,319    |
| `freight_cost_usd`| 6,198           | 8,538        | +2,340    |

Recovery rate: **~36% of previously null weight values were recovered** without dropping a single row.

### Unresolved References — 126 rows

A total of **126 rows** could not be resolved. Investigation shows these 
references point to IDs that either:

- Do not exist in the dataset (external document references, e.g. `DN-XXXX`)
- Exist but also have a null numeric value in the referenced record

These rows retain `NaN` in `weight_kg_final` and will be handled in the 
**missing values treatment** step. They are flagged and not silently dropped.

> Note: References following the pattern `DN-XXXX` appear to point to external delivery notes outside this dataset, making resolution impossible from the available data alone.

In [24]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10324 entries, 0 to 10323
Data columns (total 47 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   ID                            10324 non-null  int64         
 1   Project Code                  10324 non-null  str           
 2   PQ #                          10324 non-null  str           
 3   PO / SO #                     10324 non-null  str           
 4   ASN/DN #                      10324 non-null  str           
 5   Country                       10324 non-null  str           
 6   Managed By                    10324 non-null  str           
 7   Fulfill Via                   10324 non-null  str           
 8   Vendor INCO Term              10324 non-null  str           
 9   Shipment Mode                 9964 non-null   str           
 10  PQ First Sent to Client Date  10324 non-null  str           
 11  PO Sent to Vendor Date        10324 non

Final Dataset Validation

After cleaning and transformations, the dataset was validated to ensure correct data types and consistency across key analytical fields.

In [25]:
df.to_csv("../data/processed/shipments_clean.csv", index=False)

Exporting Clean Dataset

The cleaned dataset is exported for further analysis and modeling.